# Kịch bản Truy vấn - Máy 2 (CN002)
Áp dụng các câu truy vấn phức tạp, đa dạng (`$bucket`, `$filter`, `$unionWith`, `$facet`).

## Khởi tạo kết nối đến cả 2 máy

In [1]:
import pymongo

IP_MAY_1 = '100.68.134.19'
IP_MAY_2 = '100.118.157.52'

client_1 = pymongo.MongoClient(f'mongodb://{IP_MAY_1}:27017/', serverSelectionTimeoutMS=5000)
client_2 = pymongo.MongoClient(f'mongodb://{IP_MAY_2}:27017/', serverSelectionTimeoutMS=5000)

db_1 = client_1['fahasa_btl2']
db_2 = client_2['fahasa_btl2']
print("Đã thiết lập kết nối (client_1: Máy 1, client_2: Máy 2)")

Đã thiết lập kết nối (client_1: Máy 1, client_2: Máy 2)


## 1. Truy vấn trực tiếp tại Máy 2 
### Câu truy vấn 1: Dùng `$switch` kết hợp `$unwind` để xếp hạng và phân loại Sách bán ra tại CN002

In [2]:
pipeline_switch = [
    {"$unwind": "$ChiTiet"},
    {"$group": {
        "_id": "$ChiTiet.TenSach",
        "TongSoLuong": {"$sum": "$ChiTiet.SoLuong"}
    }},
    {"$project": {
        "_id": 0,
        "TenSach": "$_id",
        "TongSoLuong": 1,
        "PhanLoai": {
            "$switch": {
                "branches": [
                    {"case": {"$gte": ["$TongSoLuong", 500]}, "then": "🔥 Siêu HOT"},
                    {"case": {"$gte": ["$TongSoLuong", 300]}, "then": "⭐ Bán chạy"},
                    {"case": {"$gte": ["$TongSoLuong", 100]}, "then": "⚠️ Bán chậm"}
                ],
                "default": "🛑 Ế ẩm"
            }
        }
    }},
    {"$sort": {"TongSoLuong": -1}},
    {"$limit": 20}
]

print("--- Xếp hạng và phân loại Sách bán ra tại CN002 ---")
res_switch = list(db_2.hoadon.aggregate(pipeline_switch))
for r in res_switch:
    print(f"[{r['PhanLoai']}] {r['TenSach']}: Đã bán {r['TongSoLuong']} cuốn")


--- Xếp hạng và phân loại Sách bán ra tại CN002 ---
[🔥 Siêu HOT] Yes Or No - Những Quyết Định Thay Đổi Cuộc Sống: Đã bán 649 cuốn
[🔥 Siêu HOT] Lịch Sử Thế Giới Qua 6 Thức Uống - A History Of The World In 6 Glasses: Đã bán 502 cuốn
[⭐ Bán chạy] Sailor Moon - Pretty Guardian - Tập 2 (Táí Bản 2022): Đã bán 364 cuốn
[⭐ Bán chạy] Bộ Sách Vỡ Lòng Về Khoa Học - Vật Lý Học Newton Cho Trẻ Em: Đã bán 353 cuốn
[⭐ Bán chạy] Tớ Là Hải Cẩu Lông Mao - Nhà Tớ Ở Đảo Hải Cẩu: Đã bán 346 cuốn
[⭐ Bán chạy] Horrible Science - Dịch Bệnh Mắc Dịch: Đã bán 336 cuốn
[⭐ Bán chạy] One Piece - Tập 59 - Vĩnh Biệt Portgas D. Ace: Đã bán 334 cuốn
[⭐ Bán chạy] Bạch Tuyết Tóc Đỏ - Tập 8: Đã bán 332 cuốn
[⭐ Bán chạy] Boxset Sherlock Holmes Toàn Tập (Bộ 3 Cuốn): Đã bán 330 cuốn
[⭐ Bán chạy] Combo Sách Đám Trẻ Ở Đại Dương Đen + Trốn Lên Mái Nhà Để Khóc (Bộ 2 Cuốn): Đã bán 329 cuốn
[⭐ Bán chạy] Phong Thủy Đặt Mộ Và Xem Thế Đất: Đã bán 328 cuốn
[⭐ Bán chạy] Trăm Năm Nobel - Tuyển Tập Thơ William Butler Yeats - Ấn Bản Giới H

## 2. Thực hiện câu truy vấn của Máy 1 trên Máy 2 (Local)
### Câu truy vấn 2: Dùng `$filter` tìm các hóa đơn tại CN002 có chứa mặt hàng được mua với số lượng sỉ (>= 3 cuốn)

In [3]:
pipeline_filter = [
    {"$project": {
        "_id": 1,
        "MaKH": 1,
        "NgayLap": 1,
        "MatHangSi": {
            "$filter": {
                "input": "$ChiTiet",
                "as": "ct",
                "cond": {"$gte": ["$$ct.SoLuong", 3]}
            }
        }
    }},
    {"$match": {"MatHangSi.0": {"$exists": True}}}, 
    {"$limit": 5}
]

print("--- [Truy vấn Local] Hóa đơn mua sỉ tại Máy 2 (CN002) ---")
res_filter = list(db_2.hoadon.aggregate(pipeline_filter))
for r in res_filter:
    print(f"Hóa đơn: {r['_id']} | Khách hàng: {r['MaKH']}")
    for mh in r['MatHangSi']:
        print(f"  -> Sách {mh['MaSach']}: {mh['SoLuong']} cuốn")

--- [Truy vấn Local] Hóa đơn mua sỉ tại Máy 2 (CN002) ---
Hóa đơn: HD00004 | Khách hàng: KH0264
  -> Sách S0080: 3 cuốn
Hóa đơn: HD00005 | Khách hàng: KH0269
  -> Sách S0021: 3 cuốn
Hóa đơn: HD00007 | Khách hàng: KH0226
  -> Sách S0015: 3 cuốn
Hóa đơn: HD00014 | Khách hàng: KH0008
  -> Sách S0043: 3 cuốn
  -> Sách S0093: 3 cuốn
Hóa đơn: HD00019 | Khách hàng: KH0242
  -> Sách S0066: 3 cuốn


## 3. Máy 2 thực hiện truy vấn đến Máy 1 (Remote)
### Câu truy vấn 3: Dùng `$unionWith` lập danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001


In [4]:
pipeline_union = [
    # Luồng 1: Lấy danh sách Nhân viên sinh vào tháng 12
    {"$project": {
        "_id": 0, "Ma": "$_id", "Ten": "$HoTen",
        "VaiTro": {"$literal": "Nhân Viên"},
        "LienHe": "$SoDienThoai",
        "ThangSinh": {"$month": "$NgaySinh"}
    }},
    {"$match": {"ThangSinh": 12}},
    # KẾT HỢP VỚI
    {"$unionWith": {
        "coll": "khachhang",
        "pipeline": [
            # Luồng 2: Lấy danh sách Khách hàng sinh vào tháng 12
            {"$project": {
                "_id": 0, "Ma": "$_id", "Ten": "$HoTen",
                "VaiTro": {"$literal": "Khách Hàng"},
                "LienHe": "$SoDienThoai",
                "ThangSinh": {"$month": "$NgaySinh"}
            }},
            {"$match": {"ThangSinh": 12}}
        ]
    }},
    # Sắp xếp theo tên chữ cái để trộn lẫn kết quả
    {"$sort": {"Ten": 1}},
]

print("--- [Truy vấn Remote] Danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001 ---")
res_union = list(db_1.nhanvien.aggregate(pipeline_union))
for r in res_union:
    print(f"[{r['VaiTro']}] {r['Ten']} - SĐT: {r['LienHe']}")


--- [Truy vấn Remote] Danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001 ---
[Khách Hàng] Bùi Dương Thảo Vy - SĐT: 0149459979
[Nhân Viên] Bùi Quang Minh - SĐT: 0903836205
[Khách Hàng] Bùi Thái Mỹ Linh - SĐT: 0804809418
[Khách Hàng] Bùi Đức Mạnh - SĐT: 0244761543
[Khách Hàng] Bạch Hoàng Minh - SĐT: 0541718499
[Khách Hàng] Hoàng Văn Dũng - SĐT: 0177477398
[Nhân Viên] Lê Thị Ngọc Diện - SĐT: 0551377225
[Khách Hàng] Lê Tuấn Sang - SĐT: 0915562241
[Nhân Viên] Lê Vũ Chương - SĐT: 0163685716
[Khách Hàng] Nguyễn Anh Kiệt - SĐT: 0495516505
[Khách Hàng] Nguyễn Anh Tuấn - SĐT: 0862337984
[Nhân Viên] Nguyễn Minh Hoàng - SĐT: 0161147946
[Khách Hàng] Nguyễn Nhị Lam - SĐT: 0956833116
[Khách Hàng] Nguyễn Phúc Chương - SĐT: 0990553862
[Khách Hàng] Nguyễn Quốc Khánh - SĐT: 0553470836
[Khách Hàng] Nguyễn Thị Gấm - SĐT: 0826381457
[Khách Hàng] Nguyễn Thị Huyền Trân - SĐT: 0156616600
[Khách Hàng] Nguyễn Văn Khánh - SĐT: 0458010437
[Khách Hàng] Nguyễn Văn Trọng - SĐT: 0335894952
[Khách 

## 4. Thực hiện truy vấn cả 2 máy
### Câu truy vấn 4: Thống kê Tập Khách Hàng Thân Thiết Toàn Hệ Thống (Kết hợp Aggregation và Thuật toán Map-Reduce)

In [5]:
pipeline_khach_hang_trung_thanh = [
    {"$group": {
        "_id": "$MaKH",
        "SoLanMua": {"$sum": 1},
        "TongTienChiTieu": {"$sum": "$TongTien"}
    }},
    {"$lookup": {
        "from": "khachhang",
        "localField": "_id",
        "foreignField": "_id",
        "as": "ThongTinKH"
    }},
    {"$unwind": "$ThongTinKH"},
    {"$project": {
        "_id": 0,
        "MaKH": "$_id",
        "TenKH": "$ThongTinKH.HoTen",
        "LienHe": "$ThongTinKH.SoDienThoai",
        "SoLanMua": 1,
        "TongTienChiTieu": 1
    }}
]

# 1. Lấy dữ liệu thống kê từ từng Chi nhánh riêng biệt (Map)
res_1 = list(db_1.hoadon.aggregate(pipeline_khach_hang_trung_thanh))
res_2 = list(db_2.hoadon.aggregate(pipeline_khach_hang_trung_thanh))

# 2. Xử lý Gộp Dữ Liệu (Reduce) tại máy hiện tại
khach_hang_global = {}

def merge_data(res_list):
    for r in res_list:
        ma = r['MaKH']
        if ma not in khach_hang_global:
            khach_hang_global[ma] = {
                "TenKH": r['TenKH'],
                "LienHe": r['LienHe'],
                "SoLanMua": 0,
                "TongTienChiTieu": 0
            }
        khach_hang_global[ma]['SoLanMua'] += r['SoLanMua']
        khach_hang_global[ma]['TongTienChiTieu'] += r['TongTienChiTieu']

merge_data(res_1)
merge_data(res_2)

# Sắp xếp để lấy Top 5 Khách hàng thân thiết nhất 
top_kh_global = sorted(khach_hang_global.items(), key=lambda x: (x[1]['SoLanMua'], x[1]['TongTienChiTieu']), reverse=True)[:5]

# 3. Xuất Báo cáo Toàn Hệ Thống
print("========== TOP 5 KHÁCH HÀNG THÂN THIẾT NHẤT TOÀN HỆ THỐNG FAHASA ==========")
print("(Thống kê dựa trên tổng số lần mua và tổng chi tiêu ở tất cả các chi nhánh)\n")

for i, (ma_kh, info) in enumerate(top_kh_global, 1):
    print(f"⭐ TOP {i}: {info['TenKH']} ({ma_kh}) - SĐT: {info['LienHe']}")
    print(f"   ▶ Tần suất ghé hệ thống: {info['SoLanMua']} lần")
    print(f"   ▶ Tổng tiền đã chi: {info['TongTienChiTieu']:,.0f} VNĐ\n")


========== TOP 5 KHÁCH HÀNG THÂN THIẾT NHẤT TOÀN HỆ THỐNG FAHASA ==========
(Thống kê dựa trên tổng số lần mua và tổng chi tiêu ở tất cả các chi nhánh)

⭐ TOP 1: Trần Thị Thanh Thanh (KH0061) - SĐT: 0435885677
   ▶ Tần suất ghé hệ thống: 48 lần
   ▶ Tổng tiền đã chi: 53,035,000 VNĐ

⭐ TOP 2: Nguyễn Ngọc Giàu (KH0031) - SĐT: 0249690410
   ▶ Tần suất ghé hệ thống: 47 lần
   ▶ Tổng tiền đã chi: 50,830,000 VNĐ

⭐ TOP 3: Nguyễn Thị Ngọc Giàu (KH0068) - SĐT: 0355897447
   ▶ Tần suất ghé hệ thống: 47 lần
   ▶ Tổng tiền đã chi: 50,173,000 VNĐ

⭐ TOP 4: Nguyễn Hàm Thiệu (KH0186) - SĐT: 0203701261
   ▶ Tần suất ghé hệ thống: 46 lần
   ▶ Tổng tiền đã chi: 47,439,000 VNĐ

⭐ TOP 5: Bạch Hoàng Minh (KH0163) - SĐT: 0541718499
   ▶ Tần suất ghé hệ thống: 46 lần
   ▶ Tổng tiền đã chi: 45,434,000 VNĐ

